# Water Quality and Weather Data Preparation

This notebook standardizes the raw water-quality files, creates a separate weather master file, validates the outputs, and stops before any ML training.

## 1. Imports and paths

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
RAW_WATER_DIR = ROOT / 'data' / 'raw' / 'water_quality'
RAW_WEATHER_DIR = ROOT / 'data' / 'raw' / 'weather'
PROCESSED_DIR = ROOT / 'data' / 'processed'
print(ROOT)

C:\Users\AARYA PATEL\OneDrive\Desktop\Github\TechStack


## 2. Load raw data

In [2]:
water_files = sorted(RAW_WATER_DIR.glob('*.csv'))
weather_files = sorted(RAW_WEATHER_DIR.glob('*.csv'))
water_raw = {f.name: pd.read_csv(f) for f in water_files}
weather_raw = {f.name: pd.read_csv(f) for f in weather_files}
print('water files', list(water_raw))
print('weather files', list(weather_raw))

water files ['rs_session-241_as196_1.1.csv', 'RS_Session_255_AU_90.1 (1).csv', 'RS_Session_255_AU_90.2.csv']
weather files ['export (1).csv', 'export (2).csv', 'export (3).csv', 'export (4).csv', 'export (5).csv', 'export.csv']


## 3. Inspect datasets

In [3]:
for name, df in water_raw.items():
    print(name, df.shape)
    display(df.head(3))

rs_session-241_as196_1.1.csv (10, 17)


,Sr. No.,LOCATION - Water Quality Criteria,DO (mg/l) - 2011 - >5 mg/l,DO (mg/l) - 2012 - >5 mg/l,DO (mg/l) - 2013 - >5 mg/l,DO (mg/l) - 2014 - >5 mg/l,DO (mg/l) - 2015 - >5 mg/l,B.O.D (mg/l) - 2011 - < 3 mg/l,B.O.D (mg/l) - 2012 - < 3 mg/l,B.O.D (mg/l) - 2013 - < 3 mg/l,B.O.D (mg/l) - 2014 - < 3 mg/l,B.O.D (mg/l) - 2015 - < 3 mg/l,FECAL COLIFORM (MPN/100ml) - 2011 - < 2500 MPN/100ml,FECAL COLIFORM (MPN/100ml) - 2012 - < 2500 MPN/100ml,FECAL COLIFORM (MPN/100ml) - 2013 - < 2500 MPN/100ml,FECAL COLIFORM (MPN/100ml) - 2014 - < 2500 MPN/100ml,FECAL COLIFORM (MPN/100ml) - 2015 - < 2500 MPN/100ml
0,1,GANGA AT HARIDWAR D/S,6.7,7.2,6.5,5.0,9.2,5.6,5.3,5.2,5.2,2.8,1150,NaN,NaN,NaN,580
1,2,GANGA AT GARHMUKTESHWAR,8.2,8.6,9.0,9.1,7.7,3.4,3.4,2.9,2.8,3.0,1162,920.0,767.0,725.0,733
2,3,GANGA AT KANNAUJ U/S (RAJGHAT),7.9,8.6,8.2,7.8,8.4,4.5,4.0,4.0,2.8,3.5,3042,4673.0,1210.0,3500.0,2270


RS_Session_255_AU_90.1 (1).csv (93, 8)


,State,Station Name,Parameters - Dissolved Oxygen (mg/l) - (Criteria ?5.0 mg/l) - 2018,Parameters - Dissolved Oxygen (mg/l) - (Criteria ?5.0 mg/l) - 2019,Parameters - Dissolved Oxygen (mg/l) - (Criteria ?5.0 mg/l) - 2020,Parameters - Faecal Coliform (MPN/100 ml) - (Criteria ?2500 MPN/100 ml) - 2018,Parameters - Faecal Coliform (MPN/100 ml) - (Criteria ?2500 MPN/100 ml) - 2019,Parameters - Faecal Coliform (MPN/100 ml) - (Criteria ?2500 MPN/100 ml) - 2020
0,Uttarakhand,Bhagirathi at Gangotri,8.8,10.3,9.8,7.0,7.0,2.0
1,Uttarakhand,"Mandakini b/c Alaknanda, Rudraprayag",9.2,9.5,9.8,450.0,1.9,1.8
2,Uttarakhand,"Alaknanda b/c Mandakini, RudraPrayag",9.2,9.7,9.4,2300.0,1.9,1.8


RS_Session_255_AU_90.2.csv (84, 4)


,Station Code,Station Name,Parameters - Dissolved Oxygen (mg/l) - (Criteria >5.0 mg/l),Parameters - Faecal coliform (MPN/100 ml) - (Criteria <2500 MPN/100 ml)
0,Uttarakhand,"Mandakini b/c Alaknanda, Rudraprayag",10.4,1.8
1,Uttarakhand,"Alaknanda b/c Mandakini, Rudraprayag",10.2,1.8
2,Uttarakhand,"Alkananda a/c Mandakini, Rudraprayag",10.4,1.8


## 4. Standardize column names

In [4]:
print('Column names are normalized in the helper functions below before export.')

Column names are normalized in the helper functions below before export.


## 5. Normalize station/state fields

In [5]:
print({'GANGA AT HARIDWAR D/S': 'Uttarakhand', 'GANGA AT GARHMUKTESHWAR': 'Uttar Pradesh', 'GANGA AT KANNAUJ U/S (RAJGHAT)': 'Uttar Pradesh', 'GANGA AT KANNAUJ D/S, U.P': 'Uttar Pradesh', 'GANGA AT KANPUR U/S (RANIGHAT)': 'Uttar Pradesh', 'GANGA AT KANPUR D/S (JAJMAU PUMPING STATION)': 'Uttar Pradesh', 'GANGA AT ALLAHABAD D/S (SANGAM), U.P.': 'Uttar Pradesh', 'GANGA AT VARANASI D/S (MALVIYA BRIDGE), U.P': 'Uttar Pradesh', 'GANGA AT TRIGHAT (GHAZIPUR)': 'Uttar Pradesh', 'GANGA AT DAKSHINESHWAR': 'West Bengal'})

{'GANGA AT HARIDWAR D/S': 'Uttarakhand', 'GANGA AT GARHMUKTESHWAR': 'Uttar Pradesh', 'GANGA AT KANNAUJ U/S (RAJGHAT)': 'Uttar Pradesh', 'GANGA AT KANNAUJ D/S, U.P': 'Uttar Pradesh', 'GANGA AT KANPUR U/S (RANIGHAT)': 'Uttar Pradesh', 'GANGA AT KANPUR D/S (JAJMAU PUMPING STATION)': 'Uttar Pradesh', 'GANGA AT ALLAHABAD D/S (SANGAM), U.P.': 'Uttar Pradesh', 'GANGA AT VARANASI D/S (MALVIYA BRIDGE), U.P': 'Uttar Pradesh', 'GANGA AT TRIGHAT (GHAZIPUR)': 'Uttar Pradesh', 'GANGA AT DAKSHINESHWAR': 'West Bengal'}


## 6. Convert yearly wide data to long format

In [6]:
print('Water file 1 is expanded from wide yearly columns into long rows; files 2 and 3 are already year-wise or yearless and are standardized separately.')

Water file 1 is expanded from wide yearly columns into long rows; files 2 and 3 are already year-wise or yearless and are standardized separately.


## 7. Handle missing values WITHOUT inventing values

In [7]:
print('Missing values are preserved. No interpolation, backfilling, or year guessing is used.')

Missing values are preserved. No interpolation, backfilling, or year guessing is used.


## 8. Standardize datatypes

In [8]:
print('Numeric columns are cast with pandas.to_numeric; year uses nullable Int64.')

Numeric columns are cast with pandas.to_numeric; year uses nullable Int64.


## 9. Combine the three water-quality datasets

In [9]:
water_master = pd.read_csv(PROCESSED_DIR / 'water_quality_master.csv')
water_master.head()

,state,station_name,year,dissolved_oxygen,bod,fecal_coliform,source_file
0,Uttarakhand,GANGA AT HARIDWAR D/S,2011.0,6.7,5.6,1150.0,rs_session-241_as196_1.1.csv
1,Uttarakhand,GANGA AT HARIDWAR D/S,2012.0,7.2,5.3,NaN,rs_session-241_as196_1.1.csv
2,Uttarakhand,GANGA AT HARIDWAR D/S,2013.0,6.5,5.2,NaN,rs_session-241_as196_1.1.csv
3,Uttarakhand,GANGA AT HARIDWAR D/S,2014.0,5.0,5.2,NaN,rs_session-241_as196_1.1.csv
4,Uttarakhand,GANGA AT HARIDWAR D/S,2015.0,9.2,2.8,580.0,rs_session-241_as196_1.1.csv


## 10. Validate the combined dataset

In [10]:
print('rows', len(water_master))
print('stations', water_master['station_name'].nunique())
print('states', water_master['state'].nunique(dropna=True))
print('year coverage', water_master['year'].dropna().astype(int).sort_values().unique().tolist())
print('missing values')
display(water_master.isna().sum())
print('duplicate rows', water_master.duplicated().sum())
print('duplicate station-year', water_master.dropna(subset=['year']).duplicated(subset=['state', 'station_name', 'year']).sum())
print('source distribution')
display(water_master['source_file'].value_counts())

rows 413
stations 107
states 5
year coverage [2011, 2012, 2013, 2014, 2015, 2018, 2019, 2020]
missing values


state                 0
station_name          0
year                 84
dissolved_oxygen      3
bod                 363
fecal_coliform       31
source_file           0
dtype: int64

duplicate rows 0
duplicate station-year 0
source distribution


source_file
RS_Session_255_AU_90.1 (1).csv    279
RS_Session_255_AU_90.2.csv         84
rs_session-241_as196_1.1.csv       50
Name: count, dtype: int64

## 11. Export processed dataset

In [11]:
print('water_quality_master.csv and weather_master.csv are written to data/processed/')

water_quality_master.csv and weather_master.csv are written to data/processed/


## Weather master

In [12]:
weather_master = pd.read_csv(PROCESSED_DIR / 'weather_master.csv')
print(weather_master.shape)
display(weather_master.head())

(1152, 14)


,location,time,temperature,dew_point,humidity,precipitation,snow,wind_direction,wind_speed,wind_gust,pressure,sunshine_duration,weather_code,source_file
0,Haridwar,2026-08-07 00:30:00,27.1,26.4,96,0.0,NaN,165,4.0,NaN,1000.5,NaN,3,export (1).csv
1,Haridwar,2026-08-07 01:30:00,27.0,26.5,97,0.2,NaN,88,2.9,NaN,999.9,NaN,7,export (1).csv
2,Haridwar,2026-08-07 02:30:00,26.7,26.4,98,0.0,NaN,83,2.9,NaN,999.7,NaN,3,export (1).csv
3,Haridwar,2026-08-07 03:30:00,26.6,26.3,98,0.0,NaN,97,2.9,NaN,999.3,NaN,3,export (1).csv
4,Haridwar,2026-08-07 04:30:00,26.5,26.2,98,0.1,NaN,125,2.2,NaN,999.2,NaN,17,export (1).csv


### Handoff note
Historical water-quality records and weather observations are standardized separately. They are not merged here because date compatibility has not been established.